# Training Dataset Difference Analysis
Analyze differences between original and edited images to create better masks

In [ ]:
# Imports
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

# For better diff analysis
from skimage import filters, morphology, measure, segmentation
from scipy import ndimage

print("Libraries imported successfully!")

In [ ]:
# Load the training dataset
train_csv = "/sc/home/felix.boelter/recreategoods/qwen-image-edit-finetune/data/example_image_dataset/train.csv"
df_train = pd.read_csv(train_csv)

print(f"Training dataset: {len(df_train)} samples")
print("\nColumns:", df_train.columns.tolist())
print("\nFirst 3 samples:")
df_train.head(3)

In [ ]:
# Check the structure of the data
print("Sample paths:")
for i in range(min(3, len(df_train))):
    row = df_train.iloc[i]
    print(f"Row {i}:")
    for col in df_train.columns:
        if 'image' in col.lower() or 'path' in col.lower():
            print(f"  {col}: {row[col]}")
    print()

In [ ]:
# Define difference analysis functions
def compute_image_diff(img1_path, img2_path, method='lab_delta'):
    """
    Compute difference between two images
    Methods: 'simple', 'lab_delta', 'perceptual'
    """
    # Load images
    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)
    
    if img1 is None or img2 is None:
        return None
    
    # Resize to same dimensions if needed
    if img1.shape != img2.shape:
        h, w = min(img1.shape[0], img2.shape[0]), min(img1.shape[1], img2.shape[1])
        img1 = cv2.resize(img1, (w, h))
        img2 = cv2.resize(img2, (w, h))
    
    if method == 'simple':
        # Simple RGB difference
        diff = cv2.absdiff(img1, img2)
        diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        
    elif method == 'lab_delta':
        # LAB color space difference (more perceptual)
        lab1 = cv2.cvtColor(img1, cv2.COLOR_BGR2LAB)
        lab2 = cv2.cvtColor(img2, cv2.COLOR_BGR2LAB)
        
        # Delta E calculation (simplified)
        diff_lab = lab1.astype(np.float32) - lab2.astype(np.float32)
        diff_gray = np.sqrt(np.sum(diff_lab**2, axis=2))
        diff_gray = (diff_gray / diff_gray.max() * 255).astype(np.uint8)
        
    elif method == 'perceptual':
        # Convert to grayscale and use structural differences
        gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
        
        # SSIM-based difference
        from skimage.metrics import structural_similarity as ssim
        ssim_score, diff_gray = ssim(gray1, gray2, full=True)
        diff_gray = (1 - diff_gray) * 255
        diff_gray = diff_gray.astype(np.uint8)
    
    return diff_gray

def create_mask_from_diff(diff_gray, threshold=30, morphology_kernel=5):
    """
    Create a binary mask from difference image
    """
    # Threshold to create binary mask
    _, mask = cv2.threshold(diff_gray, threshold, 255, cv2.THRESH_BINARY)
    
    # Apply morphological operations to clean up
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morphology_kernel, morphology_kernel))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # Fill holes
    mask = ndimage.binary_fill_holes(mask).astype(np.uint8) * 255
    
    return mask

def visualize_diff_analysis(img1_path, img2_path, prompt=""):
    """
    Visualize the difference analysis process
    """
    # Load original images
    img1 = Image.open(img1_path).convert('RGB')
    img2 = Image.open(img2_path).convert('RGB')
    
    # Compute differences
    diff_simple = compute_image_diff(img1_path, img2_path, 'simple')
    diff_lab = compute_image_diff(img1_path, img2_path, 'lab_delta')
    
    if diff_simple is None:
        print(f"Could not load images: {img1_path}, {img2_path}")
        return
    
    # Create masks
    mask_simple = create_mask_from_diff(diff_simple, threshold=30)
    mask_lab = create_mask_from_diff(diff_lab, threshold=30)
    
    # Plot results
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Row 1: Images and simple difference
    axes[0,0].imshow(img1)
    axes[0,0].set_title('Original Image')
    axes[0,0].axis('off')
    
    axes[0,1].imshow(img2)
    axes[0,1].set_title('Edited Image')
    axes[0,1].axis('off')
    
    axes[0,2].imshow(diff_simple, cmap='hot')
    axes[0,2].set_title('RGB Difference')
    axes[0,2].axis('off')
    
    axes[0,3].imshow(mask_simple, cmap='gray')
    axes[0,3].set_title('RGB Mask')
    axes[0,3].axis('off')
    
    # Row 2: LAB difference and masks
    axes[1,0].imshow(diff_lab, cmap='hot')
    axes[1,0].set_title('LAB Difference')
    axes[1,0].axis('off')
    
    axes[1,1].imshow(mask_lab, cmap='gray')
    axes[1,1].set_title('LAB Mask')
    axes[1,1].axis('off')
    
    # Overlay masks on original
    img1_array = np.array(img1) / 255.0
    mask_simple_norm = mask_simple / 255.0
    mask_lab_norm = mask_lab / 255.0
    
    overlay_simple = img1_array.copy()
    overlay_simple[:,:,0] = np.where(mask_simple_norm > 0.5, 1.0, overlay_simple[:,:,0])
    
    overlay_lab = img1_array.copy()
    overlay_lab[:,:,1] = np.where(mask_lab_norm > 0.5, 1.0, overlay_lab[:,:,1])
    
    axes[1,2].imshow(overlay_simple)
    axes[1,2].set_title('RGB Mask Overlay')
    axes[1,2].axis('off')
    
    axes[1,3].imshow(overlay_lab)
    axes[1,3].set_title('LAB Mask Overlay')
    axes[1,3].axis('off')
    
    plt.suptitle(f'Difference Analysis\nPrompt: {prompt[:60]}...', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    coverage_simple = np.mean(mask_simple > 0) * 100
    coverage_lab = np.mean(mask_lab > 0) * 100
    
    print(f"Mask coverage - RGB: {coverage_simple:.1f}%, LAB: {coverage_lab:.1f}%")
    
    return diff_simple, diff_lab, mask_simple, mask_lab

print("Difference analysis functions defined!")

In [ ]:
# Test on a few samples
n_samples = 3

for i in range(min(n_samples, len(df_train))):
    row = df_train.iloc[i]
    
    # Get image paths (adjust column names based on your CSV structure)
    # You'll need to check what the actual column names are
    print(f"\n{'='*60}")
    print(f"Sample {i+1}:")
    print(f"Row data: {dict(row)}")
    
    # Try to identify image columns
    image_cols = [col for col in df_train.columns if 'image' in col.lower() or 'path' in col.lower()]
    print(f"Image columns found: {image_cols}")
    
    if len(image_cols) >= 2:
        original_path = row[image_cols[0]]
        edited_path = row[image_cols[1]]
        
        # Check if paths exist
        if os.path.exists(original_path) and os.path.exists(edited_path):
            prompt = row.get('prompt', row.get('text', row.get('caption', 'No prompt found')))
            print(f"Prompt: {prompt}")
            
            visualize_diff_analysis(original_path, edited_path, str(prompt))
        else:
            print(f"Missing files: {original_path}, {edited_path}")
    else:
        print("Could not identify image columns")
        break

In [ ]:
# Batch process and save difference masks
def process_training_diffs(df, output_dir="training_diff_masks", max_samples=20):
    """
    Process training dataset to create difference masks
    """
    os.makedirs(output_dir, exist_ok=True)
    
    results = []
    image_cols = [col for col in df.columns if 'image' in col.lower() or 'path' in col.lower()]
    
    if len(image_cols) < 2:
        print(f"Need at least 2 image columns, found: {image_cols}")
        return None
    
    for i in range(min(max_samples, len(df))):
        row = df.iloc[i]
        
        original_path = row[image_cols[0]]
        edited_path = row[image_cols[1]]
        
        if not (os.path.exists(original_path) and os.path.exists(edited_path)):
            continue
            
        try:
            # Compute differences
            diff_lab = compute_image_diff(original_path, edited_path, 'lab_delta')
            
            if diff_lab is None:
                continue
                
            # Create masks with different thresholds
            mask_strict = create_mask_from_diff(diff_lab, threshold=50)  # High threshold
            mask_medium = create_mask_from_diff(diff_lab, threshold=30)  # Medium threshold
            mask_loose = create_mask_from_diff(diff_lab, threshold=15)   # Low threshold
            
            # Save masks
            base_name = f"sample_{i:04d}"
            
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_diff.png"), diff_lab)
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_mask_strict.png"), mask_strict)
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_mask_medium.png"), mask_medium)
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_mask_loose.png"), mask_loose)
            
            # Calculate statistics
            coverage_strict = np.mean(mask_strict > 0) * 100
            coverage_medium = np.mean(mask_medium > 0) * 100
            coverage_loose = np.mean(mask_loose > 0) * 100
            
            prompt = row.get('prompt', row.get('text', row.get('caption', 'No prompt')))
            
            results.append({
                'sample_idx': i,
                'original_path': original_path,
                'edited_path': edited_path,
                'prompt': str(prompt),
                'diff_path': os.path.join(output_dir, f"{base_name}_diff.png"),
                'mask_strict_path': os.path.join(output_dir, f"{base_name}_mask_strict.png"),
                'mask_medium_path': os.path.join(output_dir, f"{base_name}_mask_medium.png"),
                'mask_loose_path': os.path.join(output_dir, f"{base_name}_mask_loose.png"),
                'coverage_strict': coverage_strict,
                'coverage_medium': coverage_medium,
                'coverage_loose': coverage_loose,
                'diff_mean': float(np.mean(diff_lab)),
                'diff_std': float(np.std(diff_lab))
            })
            
            if i % 5 == 0:
                print(f"Processed {i+1}/{min(max_samples, len(df))} samples")
                
        except Exception as e:
            print(f"Error processing sample {i}: {e}")
            continue
    
    # Save results
    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(output_dir, 'diff_analysis_results.csv'), index=False)
    
    print(f"\nProcessed {len(results)} samples")
    print(f"Results saved to {output_dir}")
    
    return results_df

# Run the batch processing
diff_results = process_training_diffs(df_train, max_samples=10)
if diff_results is not None:
    diff_results.head()

In [ ]:
# Analyze the results
if diff_results is not None and len(diff_results) > 0:
    print("Difference Mask Analysis Results:")
    print(f"Total samples processed: {len(diff_results)}")
    
    print("\nMask Coverage Statistics:")
    for threshold in ['strict', 'medium', 'loose']:
        col = f'coverage_{threshold}'
        print(f"\n{threshold.upper()} threshold:")
        print(f"  Mean coverage: {diff_results[col].mean():.1f}%")
        print(f"  Std coverage: {diff_results[col].std():.1f}%")
        print(f"  Min coverage: {diff_results[col].min():.1f}%")
        print(f"  Max coverage: {diff_results[col].max():.1f}%")
    
    print("\nDifference Image Statistics:")
    print(f"  Mean diff intensity: {diff_results['diff_mean'].mean():.1f}")
    print(f"  Mean diff variation: {diff_results['diff_std'].mean():.1f}")
    
    # Plot coverage distribution
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for i, threshold in enumerate(['strict', 'medium', 'loose']):
        col = f'coverage_{threshold}'
        axes[i].hist(diff_results[col], bins=10, alpha=0.7)
        axes[i].set_title(f'{threshold.upper()} Threshold Coverage')
        axes[i].set_xlabel('Coverage %')
        axes[i].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()
else:
    print("No results to analyze")

In [ ]:
# Compare difference masks with CLIPSeg masks
def compare_diff_vs_clipseg(sample_idx, diff_results_df, df_train):
    """
    Compare difference-based masks with CLIPSeg masks for the same prompt
    """
    if sample_idx >= len(diff_results_df):
        print(f"Sample {sample_idx} not available")
        return
        
    result_row = diff_results_df.iloc[sample_idx]
    train_row = df_train.iloc[result_row['sample_idx']]
    
    # Load images
    original_img = Image.open(result_row['original_path']).convert('RGB')
    diff_mask = cv2.imread(result_row['mask_medium_path'], cv2.IMREAD_GRAYSCALE)
    
    # Get CLIPSeg mask (reuse previous functions)
    # You'll need to import the CLIPSeg functions from the previous notebook
    
    prompt = result_row['prompt']
    print(f"Prompt: {prompt}")
    
    # Extract parts and create CLIPSeg mask
    # This would use the functions from your CLIPSeg notebook
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(original_img)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(diff_mask, cmap='gray')
    axes[1].set_title('Difference Mask')
    axes[1].axis('off')
    
    # For now, just show the difference mask
    # You can add CLIPSeg comparison later
    
    plt.suptitle(f'Comparison for: {prompt[:50]}...', fontsize=12)
    plt.tight_layout()
    plt.show()

# Test the comparison
if diff_results is not None and len(diff_results) > 0:
    compare_diff_vs_clipseg(0, diff_results, df_train)